## 1. Load & Filter

In [1]:
import pandas as pd
import numpy as np
import json

results_df = pd.read_csv('data/results_clean.csv', parse_dates=['date'])
shootouts_df = pd.read_csv('data/shootouts_clean.csv', parse_dates=['date'])
goalscorers_df = pd.read_csv('data/goalscorers_clean.csv', parse_dates=['date'])

print(f'Total partidos: {len(results_df)}')
print(f'Torneos únicos: {results_df["tournament"].nunique()}')

Total partidos: 11177
Torneos únicos: 117


In [2]:
# Filtrar amistosos — solo torneos oficiales
FRIENDLY_KEYWORDS = ['friendly', 'invitational', 'tour', 'camp']

def is_friendly(tournament):
    t = str(tournament).lower()
    return any(kw in t for kw in FRIENDLY_KEYWORDS)

official_df = results_df[~results_df['tournament'].apply(is_friendly)].copy()
official_df = official_df.sort_values('date').reset_index(drop=True)

print(f'Partidos oficiales: {len(official_df)}')
print(f'Partidos filtrados (amistosos): {len(results_df) - len(official_df)}')
print(f'\nTorneos oficiales únicos:')
print(sorted(official_df['tournament'].unique()))

Partidos oficiales: 9146
Partidos filtrados (amistosos): 2031

Torneos oficiales únicos:
['AFC Asian Cup', 'AFC Asian Cup qualification', 'AFC Championship', 'AFC Olympic qualification', 'AFF Championship', "AFF Women's Cup", 'ASEAN Championship', 'Africa Cup of Nations qualification', 'African Championship', 'African Championship qualification', 'African Cup of Nations', 'African Cup of Nations qualification', 'African Games', 'Aisha Buhari Cup', 'Algarve Cup', 'Aphrodite Cup', 'Arab Championship', 'Arab Cup', 'Arnold Clark Cup', 'Asian Games', 'Australian Cup', 'Baltic Cup', 'Bangladesh Tri-Nations Cup', 'CAFA Championship', 'CECAFA Championship', 'CONCACAF Championship', 'CONCACAF Championship qualification', 'CONCACAF Gold Cup', 'CONCACAF Gold Cup qualification', 'CONIFA World Cup', 'COSAFA Championship', 'Central American Games', 'Central American and Caribbean Games', 'Chunghua Cup', 'Copa América', 'Cup of Nations', 'Cyprus Cup', 'EAFF Championship', 'EAFF E-1 Championship', 'EA

## 2. Elo Ratings (solo partidos oficiales)

In [3]:
K = 40
HOME_ADVANTAGE = 100  # bonus Elo para local (ignorado si neutral=True)

elo_ratings = {}

def get_elo(team):
    return elo_ratings.get(team, 1500)

def expected_score(Ra, Rb):
    return 1 / (1 + 10 ** ((Rb - Ra) / 400))

def update_elo(home, away, home_score, away_score, neutral):
    Ra = get_elo(home)
    Rb = get_elo(away)

    # Aplicar ventaja de local solo si no es cancha neutral
    Ra_adj = Ra if neutral else Ra + HOME_ADVANTAGE

    E_a = expected_score(Ra_adj, Rb)
    E_b = 1 - E_a

    if home_score > away_score:
        S_a, S_b = 1, 0
    elif home_score < away_score:
        S_a, S_b = 0, 1
    else:
        S_a, S_b = 0.5, 0.5  # empate o penales = draw para Elo

    elo_ratings[home] = Ra + K * (S_a - E_a)
    elo_ratings[away] = Rb + K * (S_b - E_b)

# Registrar Elo pre-partido en el df oficial
official_df['elo_home'] = np.nan
official_df['elo_away'] = np.nan

for i, row in official_df.iterrows():
    official_df.at[i, 'elo_home'] = get_elo(row['home_team'])
    official_df.at[i, 'elo_away'] = get_elo(row['away_team'])
    update_elo(row['home_team'], row['away_team'],
               row['home_score'], row['away_score'],
               row['neutral'])

official_df['elo_diff'] = official_df['elo_home'] - official_df['elo_away']

# Top 20 equipos por Elo final
elo_series = pd.Series(elo_ratings).sort_values(ascending=False)
print('Top 20 equipos por Elo:')
print(elo_series.head(20).round(1))

Top 20 equipos por Elo:
Spain            2184.7
United States    2178.3
France           2103.0
England          2078.0
Germany          2075.8
Japan            2070.1
Canada           2025.9
Sweden           2019.9
North Korea      1990.6
Brazil           1975.3
Netherlands      1943.4
Italy            1914.4
Norway           1890.4
Denmark          1870.3
Mexico           1847.3
Australia        1839.1
China PR         1822.1
Nigeria          1815.8
Colombia         1812.5
Belgium          1812.2
dtype: float64


## 3. Rolling Performance (últimos 5 partidos oficiales)

In [4]:
N = 5

# Construir historial por equipo con (gf, ga, resultado)
team_history = {}  # team -> list of (gf, ga, 'win'/'draw'/'loss')

home_rolling = []  # una fila por partido, perspectiva home
away_rolling = []  # una fila por partido, perspectiva away

for _, row in official_df.iterrows():
    home, away = row['home_team'], row['away_team']
    hg, ag = row['home_score'], row['away_score']
    mid = row['match_id']

    def get_rolling(team):
        hist = team_history.get(team, [])
        recent = hist[-N:]
        if not recent:
            return {'gf_roll': 0.0, 'ga_roll': 0.0, 'win_roll': 0.0}
        return {
            'gf_roll': np.mean([x[0] for x in recent]),
            'ga_roll': np.mean([x[1] for x in recent]),
            'win_roll': np.mean([1 if x[2] == 'win' else 0.5 if x[2] == 'draw' else 0 for x in recent])
        }

    home_stats = get_rolling(home)
    away_stats = get_rolling(away)

    home_rolling.append({'match_id': mid, **{f'home_{k}': v for k, v in home_stats.items()}})
    away_rolling.append({'match_id': mid, **{f'away_{k}': v for k, v in away_stats.items()}})

    # Actualizar historial DESPUÉS de registrar el pre-match
    home_result = 'win' if hg > ag else 'draw' if hg == ag else 'loss'
    away_result = 'win' if ag > hg else 'draw' if ag == hg else 'loss'

    team_history.setdefault(home, []).append((hg, ag, home_result))
    team_history.setdefault(away, []).append((ag, hg, away_result))

home_roll_df = pd.DataFrame(home_rolling)
away_roll_df = pd.DataFrame(away_rolling)

# Merge: un match_id tiene exactamente una fila home y una away
# Usamos drop_duplicates para evitar el bug de duplicados del notebook anterior
official_df = official_df.drop_duplicates(subset='match_id').copy()

official_df = official_df.merge(home_roll_df.drop_duplicates('match_id'), on='match_id', how='left')
official_df = official_df.merge(away_roll_df.drop_duplicates('match_id'), on='match_id', how='left')

print(f'Shape final: {official_df.shape}')
print(official_df[['match_id','home_team','away_team','home_gf_roll','home_win_roll','away_gf_roll','away_win_roll']].tail(5))

Shape final: (9146, 19)
                                    match_id           home_team   away_team  \
9141        2025-12-02_Bangladesh_Azerbaijan          Bangladesh  Azerbaijan   
9142             2025-12-02_Palestine_Jordan           Palestine      Jordan   
9143            2025-12-02_Saudi_Arabia_Iraq        Saudi Arabia        Iraq   
9144  2025-12-02_Dominican_Republic_Suriname  Dominican Republic    Suriname   
9145                 2025-12-02_Nepal_Taiwan               Nepal      Taiwan   

      home_gf_roll  home_win_roll  away_gf_roll  away_win_roll  
9141           2.6            0.7           1.0            0.6  
9142           0.6            0.3           2.2            0.6  
9143           1.8            0.5           0.8            0.2  
9144           4.4            1.0           3.0            0.6  
9145           1.0            0.5           1.6            0.4  


## 4. Head-to-Head Stats (torneos oficiales)

In [5]:
h2h_history = {}  # (teamA, teamB) sorted tuple -> list of winners
h2h_rows = []

for _, row in official_df.iterrows():
    home, away = row['home_team'], row['away_team']
    key = tuple(sorted([home, away]))
    past = h2h_history.get(key, [])

    total = len(past)
    if total > 0:
        home_wins = sum(1 for r in past if r == home)
        away_wins = sum(1 for r in past if r == away)
        draws = sum(1 for r in past if r == 'draw')
        home_h2h_winrate = home_wins / total
        away_h2h_winrate = away_wins / total
        h2h_draw_rate   = draws / total
    else:
        # Sin historial: prior neutral
        home_h2h_winrate = 0.5
        away_h2h_winrate = 0.5
        h2h_draw_rate   = 0.0

    h2h_rows.append({
        'match_id': row['match_id'],
        'home_h2h_winrate': home_h2h_winrate,
        'away_h2h_winrate': away_h2h_winrate,
        'h2h_draw_rate': h2h_draw_rate,
        'h2h_total_games': total
    })

    # Actualizar historial
    if row['home_score'] > row['away_score']:
        result = home
    elif row['home_score'] < row['away_score']:
        result = away
    else:
        result = 'draw'

    h2h_history.setdefault(key, []).append(result)

h2h_df = pd.DataFrame(h2h_rows)
official_df = official_df.merge(h2h_df, on='match_id', how='left')

print(official_df[['home_team','away_team','home_h2h_winrate','away_h2h_winrate','h2h_total_games']].tail(5))

               home_team   away_team  home_h2h_winrate  away_h2h_winrate  \
9141          Bangladesh  Azerbaijan               0.5               0.5   
9142           Palestine      Jordan               0.0               1.0   
9143        Saudi Arabia        Iraq               1.0               0.0   
9144  Dominican Republic    Suriname               1.0               0.0   
9145               Nepal      Taiwan               0.5               0.5   

      h2h_total_games  
9141                0  
9142               11  
9143                1  
9144                1  
9145                0  


## 5. Función predict_match()

In [6]:
# Pesos del modelo combinado
W_ELO  = 0.50  # Elo es el predictor más robusto
W_H2H  = 0.25  # H2H captura rivalidades específicas
W_FORM = 0.25  # Forma reciente (rolling win rate)

def get_current_rolling(team):
    """Toma los últimos N partidos del historial ya construido."""
    hist = team_history.get(team, [])
    recent = hist[-N:]
    if not recent:
        return 0.5  # prior neutral
    return np.mean([1 if x[2] == 'win' else 0.5 if x[2] == 'draw' else 0 for x in recent])

def get_h2h_winrate(team_a, team_b):
    """Win rate histórico de team_a contra team_b."""
    key = tuple(sorted([team_a, team_b]))
    past = h2h_history.get(key, [])
    total = len(past)
    if total == 0:
        return 0.5  # prior neutral
    wins_a = sum(1 for r in past if r == team_a)
    return wins_a / total

def predict_match(team_a, team_b):
    """
    Retorna dict con probabilidades de victoria de team_a y team_b.
    team_a se considera 'home' a efectos de Elo (cancha neutral en bracket).
    """
    # 1. Componente Elo
    elo_a = get_elo(team_a)
    elo_b = get_elo(team_b)
    elo_prob_a = expected_score(elo_a, elo_b)  # cancha neutral
    elo_prob_b = 1 - elo_prob_a

    # 2. Componente H2H
    h2h_prob_a = get_h2h_winrate(team_a, team_b)
    h2h_prob_b = 1 - h2h_prob_a

    # 3. Componente forma reciente
    form_a = get_current_rolling(team_a)
    form_b = get_current_rolling(team_b)
    total_form = form_a + form_b
    form_prob_a = form_a / total_form if total_form > 0 else 0.5
    form_prob_b = 1 - form_prob_a

    # 4. Combinación ponderada
    raw_a = W_ELO * elo_prob_a + W_H2H * h2h_prob_a + W_FORM * form_prob_a
    raw_b = W_ELO * elo_prob_b + W_H2H * h2h_prob_b + W_FORM * form_prob_b

    # Normalizar a que sumen 1
    total = raw_a + raw_b
    prob_a = round(raw_a / total, 4)
    prob_b = round(raw_b / total, 4)

    return {
        'team_a': team_a,
        'team_b': team_b,
        'prob_a': prob_a,
        'prob_b': prob_b,
        'elo_a': round(elo_a, 1),
        'elo_b': round(elo_b, 1),
        'h2h_games': len(h2h_history.get(tuple(sorted([team_a, team_b])), [])),
        'form_a': round(form_a, 3),
        'form_b': round(form_b, 3)
    }

# Prueba rápida
test_matches = [
    ('United States', 'Germany'),
    ('Spain', 'England'),
    ('Brazil', 'France'),
    ('Sweden', 'Netherlands'),
]

for a, b in test_matches:
    r = predict_match(a, b)
    print(f"{a} vs {b}: {r['prob_a']*100:.1f}% / {r['prob_b']*100:.1f}%  (Elo: {r['elo_a']} vs {r['elo_b']})")

United States vs Germany: 70.7% / 29.3%  (Elo: 2178.3 vs 2075.8)
Spain vs England: 52.3% / 47.7%  (Elo: 2184.7 vs 2078.0)
Brazil vs France: 37.8% / 62.2%  (Elo: 1975.3 vs 2103.0)
Sweden vs Netherlands: 49.4% / 50.6%  (Elo: 2019.9 vs 1943.4)


## 6. Export predictions.json

In [ ]:
# Lista de equipos con suficientes partidos oficiales (mínimo 10)
MIN_GAMES = 10

team_game_counts = pd.concat([
    official_df['home_team'],
    official_df['away_team']
]).value_counts()

qualified_teams = sorted(team_game_counts[team_game_counts >= MIN_GAMES].index.tolist())
print(f'Equipos calificados (>= {MIN_GAMES} partidos oficiales): {len(qualified_teams)}')

# Construir objeto de equipos
teams_data = {}
for team in qualified_teams:
    teams_data[team] = {
        'elo': round(get_elo(team), 1),
        'form': round(get_current_rolling(team), 3),
        'official_games': int(team_game_counts.get(team, 0))
    }

# Construir todas las predicciones H2H entre equipos calificados
# (solo generamos el dict de lookup, la app pedirá los pares que necesite)
# Para el JSON exportamos solo los top teams para no hacer el archivo gigante
TOP_N = 50  # top N equipos por Elo para el bracket

top_teams = sorted(teams_data.keys(), key=lambda t: teams_data[t]['elo'], reverse=True)[:TOP_N]

matchups = {}
for i, ta in enumerate(top_teams):
    for tb in top_teams[i+1:]:
        key = f"{ta}|{tb}"
        matchups[key] = predict_match(ta, tb)

print(f'Matchups calculados: {len(matchups)}')

# Estructura final del JSON
output = {
    'meta': {
        'generated_at': pd.Timestamp.now().isoformat(),
        'total_official_matches': len(official_df),
        'model_weights': {'elo': W_ELO, 'h2h': W_H2H, 'form': W_FORM},
        'rolling_window': N
    },
    'teams': teams_data,
    'top_teams': top_teams,
    'matchups': matchups
}

with open('womens-football-app/public/predictions.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print('\n✅ predictions.json exportado correctamente')
print(f'Tamaño: {len(json.dumps(output)) / 1024:.1f} KB')

Equipos calificados (>= 10 partidos oficiales): 207
Matchups calculados: 1225

✅ predictions.json exportado correctamente
Tamaño: 233.6 KB


## 7. Verificación final

In [8]:
# Recargar el JSON y verificar estructura
with open('data/predictions.json') as f:
    loaded = json.load(f)

print('=== META ===')
print(json.dumps(loaded['meta'], indent=2))

print(f'\n=== TOP 10 EQUIPOS POR ELO ===')
top10 = sorted(loaded['teams'].items(), key=lambda x: x[1]['elo'], reverse=True)[:10]
for team, stats in top10:
    print(f"{team:<25} Elo: {stats['elo']:<8} Forma: {stats['form']:.3f}  Partidos: {stats['official_games']}")

print(f'\n=== EJEMPLO DE MATCHUP ===')
example_key = list(loaded['matchups'].keys())[0]
print(f'{example_key}:')
print(json.dumps(loaded['matchups'][example_key], indent=2))

=== META ===
{
  "generated_at": "2026-03-25T12:19:33.279761",
  "total_official_matches": 9146,
  "model_weights": {
    "elo": 0.5,
    "h2h": 0.25,
    "form": 0.25
  },
  "rolling_window": 5
}

=== TOP 10 EQUIPOS POR ELO ===
Spain                     Elo: 2184.7   Forma: 0.800  Partidos: 234
United States             Elo: 2178.3   Forma: 0.800  Partidos: 302
France                    Elo: 2103.0   Forma: 0.500  Partidos: 300
England                   Elo: 2078.0   Forma: 0.800  Partidos: 321
Germany                   Elo: 2075.8   Forma: 0.400  Partidos: 368
Japan                     Elo: 2070.1   Forma: 0.800  Partidos: 294
Canada                    Elo: 2025.9   Forma: 0.800  Partidos: 220
Sweden                    Elo: 2019.9   Forma: 0.200  Partidos: 427
North Korea               Elo: 1990.6   Forma: 0.800  Partidos: 157
Brazil                    Elo: 1975.3   Forma: 0.800  Partidos: 186

=== EJEMPLO DE MATCHUP ===
Spain|United States:
{
  "team_a": "Spain",
  "team_b": "United